# Bloomberg Next Open Return Predictor

This notebook ranks current S&P 500 constituents by estimated return from the selected entry price to the next regular-session open.

## BQuant-native design

- Uses **BQL only**. It does not use `blpapi`, `localhost:8194`, `xbbg`, or a Desktop API session.
- Trains immediately using historical Bloomberg `PX_LAST` as the initial 15:55 proxy and `PX_OPEN` as the next-session exit.
- When run between 15:54 and 15:59 New York time, it stores the live BQL `PX_LAST` snapshot as the exact prospective 15:55 observation.
- On later runs, stored exact snapshots automatically replace the close proxy for matching sessions.
- Reports the number of exact versus proxy observations. It never describes closing-price history as exact 15:55 history.

Run all cells in order. For an exact live entry ranking, run at approximately **15:55 New York time**.

In [14]:
# Cell 1 — Imports and configuration
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

try:
    import bql
except ImportError as exc:
    raise ImportError("Run this notebook inside Bloomberg BQuant; the bql package is required.") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
NY = "America/New_York"

CFG = {
    "index": "SPX Index",
    "benchmark": "SPX Index",
    "history_calendar_days": 550,
    "minimum_price": 5.0,
    "minimum_train_sessions": 160,
    "test_sessions": 40,
    "top_n": 25,
    "round_trip_cost_bp": 12.0,
    "minimum_net_return_bp": 10.0,
    "maximum_abs_target": 0.25,
    "winsor_quantiles": (0.005, 0.995),
    "capture_start": "15:55",
    "capture_end": "15:55",
    "snapshot_file": Path("bquant_1555_exact_snapshots.pkl"),
}
CFG

{'index': 'SPX Index',
 'benchmark': 'SPX Index',
 'history_calendar_days': 550,
 'minimum_price': 5.0,
 'minimum_train_sessions': 160,
 'test_sessions': 40,
 'top_n': 25,
 'round_trip_cost_bp': 12.0,
 'minimum_net_return_bp': 10.0,
 'maximum_abs_target': 0.25,
 'winsor_quantiles': (0.005, 0.995),
 'capture_start': '15:55',
 'capture_end': '15:55',
 'snapshot_file': PosixPath('bquant_1555_exact_snapshots.pkl')}

In [15]:
# Cell 2 — BQL helpers and current S&P 500 metadata
bq = bql.Service()

def bql_frame(response, labels):
    # Combine the BQL response items individually using their DataFrames.
    items = list(response)
    if len(items) != len(labels):
        raise RuntimeError(f"BQL returned {len(items)} items for {len(labels)} requested labels: {labels}")

    frames = []
    ignored = {"currency", "revision_date", "as_of_date", "period_end_date"}
    for label, item in zip(labels, items):
        raw = item.df().reset_index()
        raw = raw.rename(columns={c: str(c).strip().lower() for c in raw.columns})
        raw = raw.rename(columns={"id": "ticker"})

        dimensions = [c for c in ["ticker", "date"] if c in raw.columns]
        if "ticker" not in dimensions:
            raise KeyError(f"BQL item '{label}' has no ID/ticker column. Returned: {raw.columns.tolist()}")

        if label in raw.columns:
            value_col = label
        else:
            candidates = [c for c in raw.columns if c not in dimensions and c not in ignored]
            if len(candidates) != 1:
                raise KeyError(
                    f"Cannot identify value column for BQL item '{label}'. "
                    f"Candidates={candidates}; returned={raw.columns.tolist()}"
                )
            value_col = candidates[0]

        frame = raw[dimensions + [value_col]].rename(columns={value_col: label})
        frame = frame.drop_duplicates(dimensions, keep="last")
        frames.append(frame)

    out = frames[0]
    for frame in frames[1:]:
        keys = [c for c in ["ticker", "date"] if c in out.columns and c in frame.columns]
        if not keys:
            raise RuntimeError(f"BQL items cannot be joined; columns are {out.columns.tolist()} and {frame.columns.tolist()}")
        out = out.merge(frame, on=keys, how="outer", validate="one_to_one")
    return out

members = bq.univ.members(CFG["index"])
meta_items = {
    "name": bq.data.name(),
    "sector": bq.data.gics_sector_name(),
    "live_px_last": bq.data.px_last(),
}
meta_captured_at_ny = pd.Timestamp.now(tz=NY)
meta_response = bq.execute(bql.Request(members, meta_items))
meta = bql_frame(meta_response, ["name", "sector", "live_px_last"])
meta = meta[["ticker", "name", "sector", "live_px_last"]].drop_duplicates("ticker")
meta["live_px_last"] = pd.to_numeric(meta["live_px_last"], errors="coerce")
if len(meta) < 400:
    raise RuntimeError(f"BQL returned only {len(meta)} constituents; expected at least 400.")

benchmark_live_response = bq.execute(bql.Request(CFG["benchmark"], {"live_px_last": bq.data.px_last()}))
benchmark_live = bql_frame(benchmark_live_response, ["live_px_last"])[["ticker", "live_px_last"]]
benchmark_live["live_px_last"] = pd.to_numeric(benchmark_live["live_px_last"], errors="coerce")
live_snapshot_meta = pd.concat([meta[["ticker", "live_px_last"]], benchmark_live], ignore_index=True)
live_snapshot_meta = live_snapshot_meta.dropna(subset=["live_px_last"]).drop_duplicates("ticker", keep="last")
print(f"Current eligible universe: {len(meta):,} stocks")
display(meta.head())

Current eligible universe: 503 stocks


,ticker,name,sector,live_px_last
0,A UN Equity,Agilent Technologies Inc,Health Care,156.300003
1,AAPL UW Equity,Apple Inc,Information Technology,311.299988
2,ABBV UN Equity,AbbVie Inc,Health Care,261.829987
3,ABNB UW Equity,Airbnb Inc,Consumer Discretionary,185.000000
4,ABT UN Equity,Abbott Laboratories,Health Care,114.139999


In [16]:
# Cell 3 — Historical Bloomberg PX_LAST and PX_OPEN through BQL
now_ny = pd.Timestamp.now(tz=NY)
end_date = now_ny.date().isoformat()
start_date = (now_ny.normalize() - pd.Timedelta(days=CFG["history_calendar_days"])).date().isoformat()
date_range = bq.func.range(start_date, end_date)

history_items = {
    "px_last": bq.data.px_last(dates=date_range),
    "px_open": bq.data.px_open(dates=date_range),
}

equity_response = bq.execute(bql.Request(members, history_items))
equity_history = bql_frame(equity_response, ["px_last", "px_open"])

# Pull the benchmark separately; relying on index membership would omit the index itself.
benchmark_response = bq.execute(bql.Request(CFG["benchmark"], history_items))
benchmark_history = bql_frame(benchmark_response, ["px_last", "px_open"])

history = pd.concat([equity_history, benchmark_history], ignore_index=True)
history["date"] = pd.to_datetime(history["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
for c in ["px_last", "px_open"]:
    history[c] = pd.to_numeric(history[c], errors="coerce")
history = history.dropna(subset=["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")
history = history.sort_values(["ticker", "date"]).reset_index(drop=True)

if history.empty:
    raise RuntimeError("BQL returned no historical price data.")
if history["date"].nunique() < CFG["minimum_train_sessions"] + CFG["test_sessions"] + 25:
    raise RuntimeError(f"Only {history['date'].nunique()} sessions returned; increase history or check BQL entitlements.")
print(f"History: {history['date'].min().date()} to {history['date'].max().date()}, {len(history):,} rows")

History: 2025-02-15 to 2026-08-20, 278,208 rows


In [17]:
# Cell 4 — Capture/load exact 15:55 BQL snapshots
snapshot_columns = ["ticker", "session", "entry_price", "captured_at_ny"]
if CFG["snapshot_file"].exists():
    exact_snapshots = pd.read_pickle(CFG["snapshot_file"])
    missing = [c for c in snapshot_columns if c not in exact_snapshots.columns]
    if missing:
        raise KeyError(f"Snapshot file is malformed; missing columns: {missing}")
else:
    exact_snapshots = pd.DataFrame(columns=snapshot_columns)

capture_time_ny = meta_captured_at_ny
current_clock = capture_time_ny.strftime("%H:%M")
inside_capture_window = CFG["capture_start"] <= current_clock <= CFG["capture_end"]

if inside_capture_window:
    captured = live_snapshot_meta.rename(columns={"live_px_last": "entry_price"}).copy()
    captured["session"] = capture_time_ny.tz_localize(None).normalize()
    captured["captured_at_ny"] = capture_time_ny.isoformat()
    captured = captured.dropna(subset=["entry_price"])
    exact_snapshots = pd.concat([exact_snapshots, captured], ignore_index=True)
    exact_snapshots = exact_snapshots.sort_values("captured_at_ny").drop_duplicates(["ticker", "session"], keep="last")
    exact_snapshots.to_pickle(CFG["snapshot_file"])
    print(f"Captured {len(captured):,} exact 15:55 snapshots at {capture_time_ny:%Y-%m-%d %H:%M:%S %Z}.")
else:
    print(f"No exact snapshot saved: BQL snapshot time was {capture_time_ny:%H:%M}; required capture minute is {CFG['capture_start']}.")
    print("The latest ranking will use Bloomberg PX_LAST as a clearly labelled close/current-price proxy.")

exact_snapshots["session"] = pd.to_datetime(exact_snapshots["session"], errors="coerce").dt.tz_localize(None).dt.normalize()
exact_snapshots["entry_price"] = pd.to_numeric(exact_snapshots["entry_price"], errors="coerce")
print(f"Stored exact stock-session observations: {len(exact_snapshots):,}")

No exact snapshot saved: BQL snapshot time was 18:19; required capture minute is 15:55.
The latest ranking will use Bloomberg PX_LAST as a clearly labelled close/current-price proxy.
Stored exact stock-session observations: 0


In [18]:
# Cell 5 — Build exact/proxy entries, next-open targets and leakage-safe features
panel = history.rename(columns={"date": "session", "px_last": "close_proxy", "px_open": "session_open"}).copy()
panel = panel.merge(exact_snapshots[["ticker", "session", "entry_price"]], on=["ticker", "session"], how="left")
panel["entry_source"] = np.where(panel["entry_price"].notna(), "EXACT_1555", "PX_LAST_PROXY")
panel["entry_price"] = panel["entry_price"].fillna(panel["close_proxy"])
panel = panel.sort_values(["ticker", "session"]).reset_index(drop=True)

# Use the benchmark calendar to identify the exact next market session.
market_sessions = np.array(sorted(panel.loc[panel["ticker"] == CFG["benchmark"], "session"].dropna().unique()))
if len(market_sessions) < 2:
    raise RuntimeError("The benchmark did not return a usable trading calendar.")
calendar = pd.DataFrame({"session": market_sessions[:-1], "next_session": market_sessions[1:]})
panel = panel.merge(calendar, on="session", how="left")
next_opens = panel[["ticker", "session", "session_open"]].rename(columns={"session":"next_session", "session_open":"next_open"})
panel = panel.merge(next_opens, on=["ticker", "next_session"], how="left", validate="many_to_one")
panel["target_next_open"] = panel["next_open"] / panel["entry_price"] - 1

g = panel.groupby("ticker", group_keys=False)
panel["ret_intraday"] = panel["entry_price"] / panel["session_open"] - 1
panel["ret_1d"] = g["entry_price"].pct_change(1, fill_method=None)
panel["ret_5d"] = g["entry_price"].pct_change(5, fill_method=None)
panel["ret_20d"] = g["entry_price"].pct_change(20, fill_method=None)
panel["ma20_gap"] = panel["entry_price"] / g["entry_price"].transform(lambda s: s.rolling(20).mean()) - 1
panel["rv_10d"] = g["ret_1d"].transform(lambda s: s.rolling(10).std()) * np.sqrt(252)
panel["rv_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20).std()) * np.sqrt(252)

benchmark = panel[panel["ticker"] == CFG["benchmark"]][["session", "ret_intraday", "ret_1d", "ret_5d", "rv_20d"]].copy()
benchmark = benchmark.rename(columns={c: f"market_{c}" for c in benchmark.columns if c != "session"})
panel = panel.merge(benchmark, on="session", how="left", validate="many_to_one")
panel["relative_intraday"] = panel["ret_intraday"] - panel["market_ret_intraday"]
panel["relative_5d"] = panel["ret_5d"] - panel["market_ret_5d"]
for c in ["ret_intraday", "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_20d"]:
    panel[f"rank_{c}"] = panel.groupby("session")[c].rank(pct=True) - 0.5

model_df = panel[panel["ticker"] != CFG["benchmark"]].merge(meta[["ticker", "name", "sector"]], on="ticker", how="inner")
bad_target = model_df["target_next_open"].abs() > CFG["maximum_abs_target"]
model_df.loc[bad_target, "target_next_open"] = np.nan
model_df = model_df.replace([np.inf, -np.inf], np.nan)
print(f"Filtered extreme/corporate-action targets: {int(bad_target.sum()):,}")

Filtered extreme/corporate-action targets: 17


In [19]:
# Diagnostic — which feature is emptying the panel?
nan_frac = model_df[FEATURES + ["target_next_open"]].isna().mean().sort_values(ascending=False)
print("NaN fraction by column:")
print(nan_frac.to_string(), "\n")

surviving = model_df.copy()
for c in FEATURES + ["target_next_open"]:
    before = len(surviving)
    surviving = surviving.dropna(subset=[c])
    if len(surviving) < before:
        print(f"{c:<24} {before:>8,} -> {len(surviving):>8,}  (dropped {before-len(surviving):,})")

print("\n--- px_open coverage ---")
print("stocks  :", history.loc[history.ticker != CFG['benchmark'], 'px_open'].notna().mean().round(4))
bm = history[history.ticker == CFG["benchmark"]]
print("benchmark rows:", len(bm), "| px_open notna:", bm['px_open'].notna().mean().round(4),
      "| px_last notna:", bm['px_last'].notna().mean().round(4))
print("\nbenchmark sample:")
print(bm.tail(3).to_string())

NaN fraction by column:
market_rv_20d          1.000000
rv_20d                 1.000000
rv_10d                 1.000000
ma20_gap               1.000000
rank_ma20_gap          1.000000
rank_rv_20d            1.000000
rank_ret_5d            0.604950
ret_5d                 0.604950
relative_5d            0.604950
market_ret_5d          0.601449
ret_20d                0.486591
rank_ret_20d           0.486591
target_next_open       0.468504
ret_1d                 0.468439
rank_ret_1d            0.468439
market_ret_1d          0.463768
ret_intraday           0.319366
relative_intraday      0.319366
rank_ret_intraday      0.319366
market_ret_intraday    0.313406 

ret_intraday              277,656 ->  188,982  (dropped 88,674)
ret_1d                    188,982 ->  147,590  (dropped 41,392)
ret_5d                    147,590 ->   70,797  (dropped 76,793)
ret_20d                    70,797 ->   67,281  (dropped 3,516)
ma20_gap                   67,281 ->        0  (dropped 67,281)

--- px_open co

In [20]:
# DIAGNOSTIC — run this in its own cell, ABOVE Cell 7. Do not append it to Cell 7.
FEATURES = [
    "ret_intraday", "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_10d", "rv_20d",
    "market_ret_intraday", "market_ret_1d", "market_ret_5d", "market_rv_20d",
    "relative_intraday", "relative_5d", "rank_ret_intraday", "rank_ret_1d",
    "rank_ret_5d", "rank_ret_20d", "rank_ma20_gap", "rank_rv_20d",
]

print("=== model_df ===", model_df.shape)
missing = [c for c in FEATURES if c not in model_df.columns]
print("features missing entirely:", missing if missing else "none")

present = [c for c in FEATURES if c in model_df.columns] + ["target_next_open"]
print("\n=== NaN fraction (worst first) ===")
print(model_df[present].isna().mean().sort_values(ascending=False).round(4).to_string())

print("\n=== cumulative row loss ===")
surv = model_df.copy()
for c in present:
    b = len(surv)
    surv = surv.dropna(subset=[c])
    if len(surv) < b:
        print(f"{c:<24} {b:>8,} -> {len(surv):>8,}")
    if len(surv) == 0:
        print(f">>> PANEL EMPTIED AT: {c}")
        break

print("\n=== px_open coverage ===")
stk = history[history.ticker != CFG["benchmark"]]
bm  = history[history.ticker == CFG["benchmark"]]
print("stock px_open notna    :", round(stk["px_open"].notna().mean(), 4))
print("benchmark rows         :", len(bm))
print("benchmark px_open notna:", round(bm["px_open"].notna().mean(), 4) if len(bm) else "NO BENCHMARK ROWS")
print("benchmark px_last notna:", round(bm["px_last"].notna().mean(), 4) if len(bm) else "-")
print("\nbenchmark tail:")
print(bm.tail(3).to_string() if len(bm) else "(none)")

=== model_df === (277656, 30)
features missing entirely: none

=== NaN fraction (worst first) ===
market_rv_20d          1.0000
rv_20d                 1.0000
rv_10d                 1.0000
ma20_gap               1.0000
rank_ma20_gap          1.0000
rank_rv_20d            1.0000
rank_ret_5d            0.6050
ret_5d                 0.6050
relative_5d            0.6050
market_ret_5d          0.6014
ret_20d                0.4866
rank_ret_20d           0.4866
target_next_open       0.4685
ret_1d                 0.4684
rank_ret_1d            0.4684
market_ret_1d          0.4638
ret_intraday           0.3194
relative_intraday      0.3194
rank_ret_intraday      0.3194
market_ret_intraday    0.3134

=== cumulative row loss ===
ret_intraday              277,656 ->  188,982
ret_1d                    188,982 ->  147,590
ret_5d                    147,590 ->   70,797
ret_20d                    70,797 ->   67,281
ma20_gap                   67,281 ->        0
>>> PANEL EMPTIED AT: ma20_gap

=== px_open

In [21]:
# Cell 6 — Data audit and latest scoring session
session_counts = model_df.groupby("session")["entry_price"].count()
minimum_coverage_count = int(np.ceil(0.85 * meta["ticker"].nunique()))
eligible_sessions = session_counts.index[session_counts >= minimum_coverage_count]
if len(eligible_sessions) == 0:
    raise RuntimeError("No session has at least 85% universe coverage.")
latest_session = eligible_sessions.max()
latest = model_df[model_df["session"] == latest_session].copy()

audit = pd.Series({
    "latest_scoring_session": latest_session,
    "eligible_current_constituents": meta["ticker"].nunique(),
    "latest_entry_observations": latest["entry_price"].notna().sum(),
    "latest_exact_1555_observations": (latest["entry_source"] == "EXACT_1555").sum(),
    "all_exact_1555_observations": (model_df["entry_source"] == "EXACT_1555").sum(),
    "all_proxy_observations": (model_df["entry_source"] == "PX_LAST_PROXY").sum(),
    "duplicate_ticker_sessions": model_df.duplicated(["ticker", "session"]).sum(),
    "nonpositive_entry_prices": (model_df["entry_price"].dropna() <= 0).sum(),
})
display(audit.to_frame("value"))

if audit["duplicate_ticker_sessions"]:
    raise AssertionError("Duplicate ticker-session observations detected.")
if audit["nonpositive_entry_prices"]:
    raise AssertionError("Non-positive entry prices detected.")
if latest_session < now_ny.tz_localize(None).normalize() - pd.Timedelta(days=5):
    raise RuntimeError("Latest BQL scoring session is stale by more than five calendar days.")

,value
latest_scoring_session,2026-08-20 00:00:00
eligible_current_constituents,503
latest_entry_observations,503
latest_exact_1555_observations,0
all_exact_1555_observations,0
all_proxy_observations,277656
duplicate_ticker_sessions,0
nonpositive_entry_prices,0


In [22]:
# Cell 7 — Expanding-window, out-of-sample model evaluation
FEATURES = [
    "ret_intraday", "ret_1d", "ret_5d", "ret_20d", "ma20_gap", "rv_10d", "rv_20d",
    "market_ret_intraday", "market_ret_1d", "market_ret_5d", "market_rv_20d",
    "relative_intraday", "relative_5d", "rank_ret_intraday", "rank_ret_1d",
    "rank_ret_5d", "rank_ret_20d", "rank_ma20_gap", "rank_rv_20d",
]

labelled = model_df.dropna(subset=FEATURES + ["target_next_open"]).copy()
sessions = np.array(sorted(labelled["session"].unique()))
required = CFG["minimum_train_sessions"] + CFG["test_sessions"]
if len(sessions) < required:
    raise RuntimeError(f"Only {len(sessions)} complete model sessions; at least {required} are required.")

def make_model():
    return HistGradientBoostingRegressor(
        loss="absolute_error", learning_rate=0.045, max_iter=180,
        max_leaf_nodes=15, min_samples_leaf=80, l2_regularization=2.0,
        random_state=42,
    )

prediction_parts = []
for test_day in sessions[-CFG["test_sessions"]:]:
    train = labelled[labelled["session"] < test_day]
    test = labelled[labelled["session"] == test_day].copy()
    if train["session"].nunique() < CFG["minimum_train_sessions"] or test.empty:
        continue
    lo, hi = train["target_next_open"].quantile(CFG["winsor_quantiles"])
    model = make_model().fit(train[FEATURES], train["target_next_open"].clip(lo, hi))
    test["prediction"] = model.predict(test[FEATURES])
    prediction_parts.append(test[["session", "ticker", "target_next_open", "prediction"]])

if not prediction_parts:
    raise RuntimeError("Walk-forward evaluation produced no predictions.")
oos = pd.concat(prediction_parts, ignore_index=True)
daily_rows = []
for day, d in oos.groupby("session", sort=True):
    n = max(1, len(d) // 10)
    daily_rows.append({
        "session": day,
        "spearman_ic": d["prediction"].corr(d["target_next_open"], method="spearman"),
        "top_decile": d.nlargest(n, "prediction")["target_next_open"].mean(),
        "bottom_decile": d.nsmallest(n, "prediction")["target_next_open"].mean(),
    })
daily = pd.DataFrame(daily_rows)
daily["long_short"] = daily["top_decile"] - daily["bottom_decile"]

metrics = pd.Series({
    "OOS sessions": oos["session"].nunique(),
    "OOS observations": len(oos),
    "MAE (bp)": mean_absolute_error(oos["target_next_open"], oos["prediction"]) * 10_000,
    "Mean daily Spearman IC": daily["spearman_ic"].mean(),
    "Positive IC session rate": (daily["spearman_ic"] > 0).mean(),
    "Mean top-decile return (bp)": daily["top_decile"].mean() * 10_000,
    "Mean top-minus-bottom (bp)": daily["long_short"].mean() * 10_000,
})
display(metrics.to_frame("value"))
(daily.set_index("session")["long_short"].fillna(0).add(1).cumprod() - 1).plot(figsize=(11,4), title="Walk-forward top-minus-bottom return (gross)")
plt.axhline(0, color="black", lw=0.8)
plt.show()

RuntimeError: Only 0 complete model sessions; at least 200 are required.

In [ ]:
# Cell 8 — Final latest-session ranking
train = labelled[labelled["session"] < latest_session].copy()
score = model_df[
    (model_df["session"] == latest_session) &
    (model_df["entry_price"] >= CFG["minimum_price"])
].dropna(subset=FEATURES).copy()
if train.empty:
    raise RuntimeError("Final model has no training observations.")
if score.empty:
    raise RuntimeError("Latest session has no complete scoring observations.")

lo, hi = train["target_next_open"].quantile(CFG["winsor_quantiles"])
final_model = make_model().fit(train[FEATURES], train["target_next_open"].clip(lo, hi))
score["predicted_gross_return"] = final_model.predict(score[FEATURES])
score["predicted_gross_bp"] = score["predicted_gross_return"] * 10_000
score["predicted_net_bp"] = score["predicted_gross_bp"] - CFG["round_trip_cost_bp"]
score["passes_hurdle"] = score["predicted_net_bp"] >= CFG["minimum_net_return_bp"]
error_sd = (oos["target_next_open"] - oos["prediction"]).std()
score["signal_to_error"] = (score["predicted_net_bp"] / 10_000) / error_sd if error_sd > 0 else np.nan
score["rank"] = score["predicted_net_bp"].rank(ascending=False, method="first").astype(int)

columns = ["rank", "ticker", "name", "sector", "session", "entry_price", "entry_source", "predicted_gross_bp", "predicted_net_bp", "signal_to_error", "passes_hurdle"]
ranking = score.sort_values("rank")[columns].reset_index(drop=True)
ranking_display = ranking.head(CFG["top_n"]).copy()
for c in ["entry_price", "predicted_gross_bp", "predicted_net_bp", "signal_to_error"]:
    ranking_display[c] = ranking_display[c].round(2)
display(ranking_display)

if (ranking["entry_source"] == "EXACT_1555").mean() < 0.85:
    print("WARNING: Latest ranking primarily uses PX_LAST proxy entries, not exact stored 15:55 snapshots.")

In [ ]:
# Cell 9 — Export results
stamp = pd.Timestamp(latest_session).strftime("%Y%m%d")
ranking_file = Path(f"Bloomberg_Next_Open_Ranking_{stamp}.csv")
diagnostics_file = Path(f"Bloomberg_Next_Open_OOS_{stamp}.csv")
ranking.to_csv(ranking_file, index=False)
oos.to_csv(diagnostics_file, index=False)
print(f"Saved ranking: {ranking_file.resolve()}")
print(f"Saved diagnostics: {diagnostics_file.resolve()}")
print(f"Stocks passing the configured net-return hurdle: {int(ranking['passes_hurdle'].sum()):,}")

## Interpretation

- `EXACT_1555` means the notebook was run in the capture window and retained that BQL snapshot.
- `PX_LAST_PROXY` means Bloomberg historical `PX_LAST` was used. It is suitable for initial research but is not an exact historical 15:55 observation.
- The model uses expanding-window validation; no future session is included in an earlier prediction.
- Current S&P 500 membership introduces survivorship bias into historical testing.
- Rankings are estimates, not guaranteed returns. Review earnings, news, liquidity, spreads and portfolio exposure before trading.